# Income Classification using Spark MLlib

The aim of this notebook is to build a machine learning classifier that predicts whether a person earns more than $50k per year using census information.

Two classifiers will be tested:

- Random Forest Classifier
- Decision Tree Classifier

In [1]:
from pyspark.sql import SparkSession

# Create a Spark session that will be used throughout the project
spark = (
    SparkSession.builder
    .appName("IncomeClassification")
    .getOrCreate()
)

# Reduce unnecessary Spark messages so the output is easier to read
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/24 16:00:10 WARN Utils: Your hostname, YA13Q50-LTE, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/24 16:00:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 16:02:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load the Dataset

The dataset is loaded into a Spark DataFrame so that it can be processed using PySpark MLlib.

The schema and sample records are inspected to understand the available variables.

In [ ]:
# Load the income dataset
df = spark.read.csv(
    "income.csv",
    header=True,
    inferSchema=True
)

# Check the data types of all columns before preprocessing
df.printSchema()

# Display a sample of the dataset
df.show(5)

# Confirm the number of records available for training
print("Total rows:", df.count())

## Identify Categorical Variables

Machine learning models require numerical inputs.

The following columns contain text values and must be converted into numerical indexes before model training.

In [4]:
from pyspark.ml.feature import StringIndexer

# These columns contain categories that must be converted to numbers
categorical_columns = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "citizenship"
]

## Convert Categorical Variables

StringIndexer is used to assign a numerical index to each category.

The target variable is also converted into a numerical label that can be used by the classifiers.

In [5]:
indexed_df = df

# Convert each categorical feature into a numerical index
for column in categorical_columns:

    # Create a numerical representation of each category
    indexer = StringIndexer(
        inputCol=column,
        outputCol=column + "_idx"
    )

    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

# Convert the income class into the prediction label
label_indexer = StringIndexer(
    inputCol="income_class",
    outputCol="label"
)

indexed_df = label_indexer.fit(indexed_df).transform(indexed_df)

indexed_df.select("income_class", "label").show(5)

## Feature Engineering

All predictor variables are combined into a single feature vector.

Spark MLlib requires a single features column containing all input variables.

In [6]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "age",
    "weight",
    "education_years",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "workclass_idx",
    "education_idx",
    "marital_status_idx",
    "occupation_idx",
    "relationship_idx",
    "race_idx",
    "sex_idx",
    "citizenship_idx"
]

# Combine all predictor variables into one feature vector
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

data = assembler.transform(indexed_df)

data.select("features", "label").show(5, truncate=False)

## Split the Dataset

The dataset is divided into training and testing sets.

The training data is used to learn patterns while the testing data is used to evaluate performance on unseen records.

In [7]:

train, test = data.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train.count())
print("Testing rows:", test.count())

## Decision Tree Classifier

A Decision Tree model is trained to classify whether a person's income is above or below $50,000.

In [8]:
from pyspark.ml.classification import DecisionTreeClassifier

# Increase maxBins so Spark can handle columns with many categories
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxBins=64
)

# Train a model that learns classification rules from the training data
dt_model = dt.fit(train)

# Apply the trained model to unseen test data
dt_predictions = dt_model.transform(test)

## Evaluate the Decision Tree Model

The model is evaluated using accuracy, precision, recall and AUC.

In [9]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

def calculate_metric(predictions, metric_name):
    """
    Calculate a classification metric for a set of predictions.

    Parameters:
        predictions : Spark DataFrame
        metric_name : Evaluation metric name

    Returns:
        Metric value
    """

    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )

    return evaluator.evaluate(predictions)

# Measure how well the model separates the income classes
dt_auc = BinaryClassificationEvaluator(
    labelCol="label"
).evaluate(dt_predictions)

print("Decision Tree Results")
print("AUC:", dt_auc)
print("Accuracy:", calculate_metric(dt_predictions, "accuracy"))
print("Precision:", calculate_metric(dt_predictions, "weightedPrecision"))
print("Recall:", calculate_metric(dt_predictions, "weightedRecall"))

## Random Forest Classifier

A Random Forest model is trained using multiple decision trees.

This approach often improves prediction accuracy and reduces overfitting.

In [10]:
from pyspark.ml.classification import RandomForestClassifier

# Use multiple trees to improve model stability and performance
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxBins=64
)

rf_model = rf.fit(train)

# Generate predictions for the test dataset
rf_predictions = rf_model.transform(test)

In [11]:
# Evaluate the Random Forest model using the same metrics
rf_auc = BinaryClassificationEvaluator(
    labelCol="label"
).evaluate(rf_predictions)

print("Random Forest Results")
print("AUC:", rf_auc)
print("Accuracy:", calculate_metric(rf_predictions, "accuracy"))
print("Precision:", calculate_metric(rf_predictions, "weightedPrecision"))
print("Recall:", calculate_metric(rf_predictions, "weightedRecall"))
